# ASL Annual-Cycle Box Statistics

Generate an ASL annual-cycle box/marker figure from the diagnostic metric data produced by `ASL_Analysis.ipynb`. This notebook mirrors the monthly Jan-Dec statistics in `scripts/ncl/asl_ancyc_gen_box_statis_hist.ncl`, using the new CSV outputs instead of intermediate NCL NetCDF files.

In [ ]:
from pathlib import Path
import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


In [ ]:
# ============================================================
# User configuration
# ============================================================

data_root = Path("/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/polar_analysis/data/asl_analysis")
figure_root = Path("/lcrc/group/e3sm/public_html/diagnostic_output/ac.szhang/polar_analysis/figure/asl_analysis/annual_cycle_box")

ts_index_path = data_root / "ts_index"
figure_root.mkdir(parents=True, exist_ok=True)

scenario = "historical"
box_group = {
    "label": "E3SMv2.1 SORRM",
    "path_pattern": "e3sm/historical/ASL.index.v2_1-SORRM.*.Monthly.1951-2014.csv",
    "color": "#4C78A8",
}

marker_groups = [
    {
        "label": "NOAA-20C",
        "path_pattern": "analysis/historical/ASL.index.NOAA_20C.en00.Monthly.1950-2014.csv",
        "color": "#F58518",
        "marker": "o",
    },
    {
        "label": "ERA5",
        "path_pattern": "analysis/historical/ASL.index.ERA5.en00.Monthly.1979-2014.csv",
        "color": "#54A24B",
        "marker": "D",
    },
]

month_numbers = list(range(1, 13))
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

plot_variables = [
    {
        "column": "lat",
        "title": "ASL Central Latitude",
        "unit": "degree",
        "ylim": (-80, -60),
        "ytick": 4,
    },
    {
        "column": "lon",
        "title": "ASL Central Longitude",
        "unit": "degree",
        "ylim": (160, 310),
        "ytick": 20,
    },
    {
        "column": "ActCenPres",
        "title": "ASL Central Pressure",
        "unit": "hPa",
        "ylim": (940, 1020),
        "ytick": 10,
    },
]

figure_file = figure_root / f"fig_pdf_ASL_{scenario}_annual_cycle_box.png"


In [ ]:
# ============================================================
# Data helpers
# ============================================================

def resolve_files(path_pattern):
    pattern = str(ts_index_path / path_pattern)
    return sorted(glob.glob(pattern))


def load_monthly_values(path_pattern, month, column):
    frames = []
    files = resolve_files(path_pattern)

    for path in files:
        df = pd.read_csv(path)

        if "month" not in df.columns:
            raise KeyError(f"Column 'month' not found in {path}")

        if column not in df.columns:
            raise KeyError(f"Column {column!r} not found in {path}")

        monthly_df = df.loc[pd.to_numeric(df["month"], errors="coerce") == month]
        values = pd.to_numeric(monthly_df[column], errors="coerce").dropna()

        if column == "lon" and len(values) and values.median() < 0:
            values = values + 360.0

        frames.append(values)

    if not frames:
        raise FileNotFoundError(
            f"No files matched for monthly input: {ts_index_path / path_pattern}"
        )

    return pd.concat(frames, ignore_index=True)


def monthly_distribution(group_config, month, column):
    return load_monthly_values(group_config["path_pattern"], month, column)


def monthly_mean(group_config, month, column):
    return float(monthly_distribution(group_config, month, column).mean())


In [ ]:
# Preview matched input files.
box_files = resolve_files(box_group["path_pattern"])
print(f"{box_group['label']} Monthly: {len(box_files)} file(s)")
for path in box_files:
    print(f"  {path}")

for group in marker_groups:
    files = resolve_files(group["path_pattern"])
    print(f"{group['label']} Monthly: {len(files)} file(s)")
    for path in files:
        print(f"  {path}")


In [ ]:
# ============================================================
# Plot annual-cycle box statistics
# ============================================================

plt.rcParams.update({
    "font.size": 12,
    "axes.linewidth": 1.2,
    "xtick.major.width": 1.2,
    "ytick.major.width": 1.2,
})

fig, axes = plt.subplots(1, len(plot_variables), figsize=(15, 4.8), constrained_layout=True)

if len(plot_variables) == 1:
    axes = [axes]

positions = np.arange(1, len(month_numbers) + 1)
panel_labels = ["(a)", "(b)", "(c)"]

for ax, variable, panel_label in zip(axes, plot_variables, panel_labels):
    column = variable["column"]
    distributions = [monthly_distribution(box_group, month, column) for month in month_numbers]

    box = ax.boxplot(
        distributions,
        positions=positions,
        widths=0.45,
        patch_artist=True,
        showfliers=False,
        medianprops={"color": "black", "linewidth": 1.4},
        boxprops={"linewidth": 1.3},
        whiskerprops={"linewidth": 1.3},
        capprops={"linewidth": 1.3},
    )

    for patch in box["boxes"]:
        patch.set_facecolor(box_group["color"])
        patch.set_alpha(0.25)

    box_means = [values.mean() for values in distributions]
    ax.scatter(
        positions,
        box_means,
        color=box_group["color"],
        edgecolor="black",
        linewidth=0.5,
        marker="s",
        s=48,
        zorder=4,
        label=f"{box_group['label']} mean",
    )

    for group in marker_groups:
        means = [monthly_mean(group, month, column) for month in month_numbers]
        ax.scatter(
            positions,
            means,
            color=group["color"],
            edgecolor="black",
            linewidth=0.5,
            marker=group["marker"],
            s=48,
            zorder=5,
            label=group["label"],
        )

    ymin, ymax = variable["ylim"]
    ax.set_ylim(ymin, ymax)
    ax.set_yticks(np.arange(ymin, ymax + 1e-9, variable["ytick"]))
    ax.set_xticks(positions)
    ax.set_xticklabels(month_labels, rotation=45, ha="right")
    ax.set_title(f"{panel_label} {variable['title']}", loc="left")
    ax.set_ylabel(f"({variable['unit']})")
    ax.grid(False)

handles, labels = axes[-1].get_legend_handles_labels()
unique = dict(zip(labels, handles))
fig.legend(
    unique.values(),
    unique.keys(),
    loc="lower center",
    bbox_to_anchor=(0.5, -0.05),
    ncol=len(unique),
    frameon=False,
)

fig.savefig(figure_file, dpi=200, bbox_inches="tight")
figure_file
